# To test the nuscenes dataloader

In [1]:
import sys
sys.path.append("..")
from nuscenes_tools.nuscenes_data import NuScenesDataset

## Define the data preprocessing pipeline

In [2]:
import nuscenes_tools.nuscenes_utils.pipelines as pipelines
data_root = '../data/nuscenes/'
mean=[0.5, 0.5, 0.5]
std=[0.5, 0.5, 0.5]
class_names = [
    'car', 'truck', 'construction_vehicle', 'bus', 'trailer', 'barrier',
    'motorcycle', 'bicycle', 'pedestrian', 'traffic_cone'
]
input_modality = dict(
    use_lidar=False,
    use_camera=True,
    use_radar=False,
    use_map=False,
    use_external=False)
data_config={
    'cams': ['CAM_FRONT_LEFT', 'CAM_FRONT', 'CAM_FRONT_RIGHT', 'CAM_BACK_LEFT', 'CAM_BACK', 'CAM_BACK_RIGHT'],
    'Ncams': 6,
    'input_size': (192, 336), 
    'src_size': (900, 1600),
    'keep_ratio': False,
    'mean': mean,
    'std': std,

    # Augmentation
    'resize': (0, 0),
    'rot': (0, 0),
    'flip': False,
    'crop_h': (0.0, 0.0),
    'resize_test':0.0,
}

# Each nuScenes sequence is ~40 keyframes long. Our training procedure samples
# sequences first, then loads frames from the sampled sequence in order 
# starting from the first frame. This reduces training step-to-step diversity,
# lowering performance. To increase diversity, we split each training sequence
# in half to ~20 keyframes, and sample these shorter sequences during training.
# During testing, we do not do this splitting.
train_sequences_split_num = 2
test_sequences_split_num = 1


train_pipeline = [
    pipelines.LoadMultiViewImageFromFiles_BEVDet(data_config, is_train=True),
    pipelines.DefaultFormatBundle3D(class_names=None, with_gt=False, with_label=False),
    pipelines.Collect3D(keys=['img_inputs']),
]

eval_pipeline = [
    pipelines.LoadMultiViewImageFromFiles_BEVDet(data_config, is_train=False),
    pipelines.DefaultFormatBundle3D(class_names=None, with_gt=False, with_label=False),
    pipelines.Collect3D(keys=['img_inputs']),
]

In [3]:
train_set = NuScenesDataset(
    data_root=data_root,
    ann_file=data_root + 'nuscenes_desc_infos_omnidrive_train.pkl',
    pipeline=train_pipeline,
    classes=class_names,
    test_mode=False,
    use_valid_flag=True,
    modality=input_modality,
    # we use box_type_3d='LiDAR' in kitti and nuscenes dataset
    # and box_type_3d='Depth' in sunrgbd and scannet dataset.
    box_type_3d='LiDAR',
    speed_mode=None,
    max_interval=None,
    min_interval=None,
    prev_only=None,
    fix_direction=None,
    img_info_prototype='bevdet',
    use_sequence_group_flag=True,
    sequences_split_num=train_sequences_split_num,
    filter_empty_gt=False
)

val_set=NuScenesDataset(pipeline=eval_pipeline, 
    classes=class_names,
    ann_file=data_root + 'nuscenes_desc_infos_omnidrive_val.pkl',
    modality=input_modality, 
    img_info_prototype='bevdet',
    use_sequence_group_flag=True,
    sequences_split_num=test_sequences_split_num,
    test_mode=True
)

## Show the meta data

In [4]:
sample = val_set[100]
print(sample.keys())
print(sample['img_metas'].data.keys())


dict_keys(['img_metas', 'img_inputs'])
dict_keys(['box_mode_3d', 'box_type_3d', 'sample_idx', 'pts_filename', 'sequence_group_idx', 'curr_to_prev_lidar_rt', 'start_of_sequence', 'global_to_curr_lidar_rt', 'prev_lidar_to_global_rt', 'sample_index', 'description', 'location', 'timeofday'])


## Visualize multiview images

The `img_inputs` item includes: image, rotations, translations, intrinsics, post_rotations (data aug), post_translations (data aug) (for each of six views)

In [5]:
for item in sample['img_inputs']:
    print(item.shape)

torch.Size([6, 3, 192, 336])
torch.Size([6, 3, 3])
torch.Size([6, 3])
torch.Size([6, 3, 3])
torch.Size([6, 3, 3])
torch.Size([6, 3])


In [6]:
import numpy as np
import PIL 

mv_imgs = sample['img_inputs'][0]

mv_imgs = mv_imgs.permute(0, 2, 3, 1).numpy()
print(mv_imgs.shape)
mv_imgs = mv_imgs * np.array(std)[None, None, None] + np.array(mean)[None, None, None]
mv_imgs = (mv_imgs * 255).astype(np.uint8)
print(mv_imgs.shape)
for i in range(mv_imgs.shape[0]):
    img = mv_imgs[i]
    print(img.shape)
    img = PIL.Image.fromarray(img)
    img.save("val_view_%d.jpg"%i)


(6, 192, 336, 3)
(6, 192, 336, 3)
(192, 336, 3)
(192, 336, 3)
(192, 336, 3)
(192, 336, 3)
(192, 336, 3)
(192, 336, 3)


## DataLoader

In [7]:
import torch
from functools import partial
from nuscenes_tools.nuscenes_utils.collate import collate

data_loader = torch.utils.data.DataLoader(
    train_set,
    batch_size=4,
    num_workers=0,
    collate_fn=partial(collate, samples_per_gpu=4),
    pin_memory=False,
)

In [8]:
for batch in data_loader:
    print(batch.keys())
    print(len(batch['img_metas'].data[0]))
    print(len(batch['img_inputs']))
    print(batch['img_inputs'][0].shape)
    break

dict_keys(['img_metas', 'img_inputs'])
4
6
torch.Size([4, 6, 3, 192, 336])


In [16]:
print(batch['img_metas'].data[0][0].keys())
print(batch['img_metas'].data[0][0]['description'])
print(batch['img_metas'].data[0][0]['location'])
print(batch['img_metas'].data[0][0]['timeofday'])


dict_keys(['box_mode_3d', 'box_type_3d', 'sample_idx', 'pts_filename', 'sequence_group_idx', 'curr_to_prev_lidar_rt', 'start_of_sequence', 'global_to_curr_lidar_rt', 'prev_lidar_to_global_rt', 'sample_index', 'description', 'location', 'timeofday'])
Car overtaking, parking lot, peds, ped exiting car, objects on the ground
boston-seaport
2018-05-21-11-06-59-0400


## Infinite length sequence

In [10]:
from nuscenes_tools.nuscenes_utils.infinite_group_each_sample_in_batch_sampler import InfiniteGroupEachSampleInBatchSampler, get_dist_info
rank, world_size = get_dist_info()

batch_size = 4
batch_sampler = InfiniteGroupEachSampleInBatchSampler(
    train_set,
    batch_size,
    world_size,
    rank,
)

seq_data_loader = torch.utils.data.DataLoader(
    train_set,
    batch_size=1,
    num_workers=4,
    batch_sampler=batch_sampler,
    collate_fn=partial(collate, samples_per_gpu=4),
    pin_memory=False,
)

seq_imgs = []
for i, seq_item in enumerate(seq_data_loader):
    meta_data = seq_item['img_metas'].data[0]
    seq_imgs.append(seq_item['img_inputs'][0])
    for meta_data_sample in meta_data:
        print(meta_data_sample['sequence_group_idx'], end=" ")
    print()
    if i > 5:
        break

844 1088 1375 50 
844 1088 1375 50 
844 1088 1375 50 
844 1088 1375 50 
844 1088 1375 50 
844 1088 1375 50 
844 1088 1375 50 


In [11]:
for timestep, batch_data in enumerate(seq_imgs):
    mv_imgs = batch_data[:, 1].permute(0, 2, 3, 1).numpy()
    mv_imgs = mv_imgs * np.array(std)[None, None, None] + np.array(mean)[None, None, None]
    mv_imgs = (mv_imgs * 255).astype(np.uint8)
    for i in range(mv_imgs.shape[0]):
        img = mv_imgs[i]
        img = PIL.Image.fromarray(img)
        img.save("sequence_vis/seq_view_batch_%d_time_%d.jpg"%(i, timestep))

## VQVAE evaluation

In [12]:
import random
import torch
torch.cuda.set_device(0)
import cv2
import numpy as np
import sys
sys.path.append('..')
from tools.run_infinity import *

model_path='../weights/infinity_125M_256x256.pth'
vae_path='../weights/infinity_vae_d16.pth'
text_encoder_ckpt = '../weights/flan-t5-xl'
args=argparse.Namespace(
    pn='0.06M',
    model_path=model_path,
    cfg_insertion_layer=0,
    vae_type=16,
    vae_path=vae_path,
    add_lvl_embeding_only_first_block=1,
    use_bit_label=1,
    model_type='infinity_layer12',
    rope2d_each_sa_layer=1,
    rope2d_normalized_by_hw=2,
    use_scale_schedule_embedding=0,
    sampling_per_bits=1,
    text_encoder_ckpt=text_encoder_ckpt,
    text_channels=2048,
    apply_spatial_patchify=0,
    h_div_w_template=7/4,
    use_flex_attn=0,
    cache_dir='/dev/shm',
    checkpoint_type='torch',
    seed=0,
    bf16=1,
    save_file='tmp.jpg',
    enable_model_cache=False,
)

/home/applied/miniconda3/envs/inf/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
mv_imgs = sample['img_inputs'][0].cuda()

vae = load_visual_tokenizer(args)
recon_mv_imgs, vq_output = vae(mv_imgs)

In [14]:
len(recon_mv_imgs)
for item in recon_mv_imgs:
    print(item.shape)

torch.Size([3, 192, 336])
torch.Size([3, 192, 336])
torch.Size([3, 192, 336])
torch.Size([3, 192, 336])
torch.Size([3, 192, 336])
torch.Size([3, 192, 336])


In [15]:
recon_mv_imgs = recon_mv_imgs.permute(0, 2, 3, 1).cpu().numpy()
recon_mv_imgs = recon_mv_imgs * np.array(std)[None, None, None] + np.array(mean)[None, None, None]
recon_mv_imgs = (recon_mv_imgs * 255).astype(np.uint8)
for i in range(recon_mv_imgs.shape[0]):
    img = recon_mv_imgs[i]
    print(img.shape)
    img = PIL.Image.fromarray(img)
    img.save("recon_val_view_%d.jpg"%i)

(192, 336, 3)
(192, 336, 3)
(192, 336, 3)
(192, 336, 3)
(192, 336, 3)
(192, 336, 3)
